# Part 1 — Model Training & Evaluation

This notebook trains all four classifiers and compares them end-to-end.

**Prerequisites**: Run `preprocessing.py` first to generate `X.npy`, `y.npy`, `label_map.npy`.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score)

import torch

NOTEBOOK_DIR = Path.cwd().resolve()
PART1_ROOT = (NOTEBOOK_DIR / "..").resolve()
DATA_ROOT = str(PART1_ROOT / "data")
ASL_DATASET = str(PART1_ROOT / "data" / "asl_dataset")
MODELS_DIR = str(PART1_ROOT / "models")
RESULTS_DIR = str(PART1_ROOT / "results")

sys.path.insert(0, str(PART1_ROOT / "src"))
from train import split_data, ASL_CNN, train_cnn, predict_cnn, load_image_data

sns.set_theme(style='whitegrid')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete")

## 1. Load Preprocessed Data

In [ ]:
X = np.load(os.path.join(DATA_ROOT, "X.npy"))
y = np.load(os.path.join(DATA_ROOT, "y.npy"))
label_map = np.load(os.path.join(DATA_ROOT, "label_map.npy"), allow_pickle=True).item()
label_names = [label_map[i] for i in range(len(label_map))]

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
print(f'Feature shape: {X_train.shape[1]}  Classes: {len(label_names)}')

## 2. SVM (RBF Kernel)

In [ ]:
svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=SEED)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)
svm_f1  = f1_score(y_test, svm_pred, average='macro', zero_division=0)
print(f'SVM  Accuracy: {svm_acc:.4f}  F1: {svm_f1:.4f}')
joblib.dump(svm, os.path.join(MODELS_DIR, "svm.pkl"))

## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1  = f1_score(y_test, rf_pred, average='macro', zero_division=0)
print(f'RF   Accuracy: {rf_acc:.4f}  F1: {rf_f1:.4f}')
joblib.dump(rf, os.path.join(MODELS_DIR, "rf.pkl"))

## 4. MLP (2-layer)

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(256, 128), activation='relu',
                    max_iter=300, early_stopping=True, random_state=SEED)
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)
mlp_acc = accuracy_score(y_test, mlp_pred)
mlp_f1  = f1_score(y_test, mlp_pred, average='macro', zero_division=0)
print(f'MLP  Accuracy: {mlp_acc:.4f}  F1: {mlp_f1:.4f}')
joblib.dump(mlp, os.path.join(MODELS_DIR, "mlp.pkl"))

## 5. CNN (on Raw Images)

In [ ]:
print("Loading image data...")
X_img, y_img, _ = load_image_data(ASL_DATASET)
Xi_train, Xi_val, Xi_test, yi_train, yi_val, yi_test = split_data(X_img, y_img)

cnn = train_cnn(Xi_train, yi_train, Xi_val, yi_val,
                num_classes=len(label_names), models_dir=MODELS_DIR)
cnn_pred = predict_cnn(cnn, Xi_test)
cnn_acc = accuracy_score(yi_test, cnn_pred)
cnn_f1  = f1_score(yi_test, cnn_pred, average='macro', zero_division=0)
print(f'CNN  Accuracy: {cnn_acc:.4f}  F1: {cnn_f1:.4f}')

In [ ]:
# Training progress is printed epoch-by-epoch above.
# To plot curves, modify train_cnn() in src/train.py to return history lists.
print("CNN training complete. Check printed epoch logs above for val_acc progression.")

## 6. Model Comparison

In [ ]:
results = [
    {'name': 'SVM',           'accuracy': svm_acc, 'f1': svm_f1,  'preds': svm_pred,  'y': y_test},
    {'name': 'Random Forest', 'accuracy': rf_acc,  'f1': rf_f1,   'preds': rf_pred,   'y': y_test},
    {'name': 'MLP',           'accuracy': mlp_acc, 'f1': mlp_f1,  'preds': mlp_pred,  'y': y_test},
    {'name': 'CNN',           'accuracy': cnn_acc, 'f1': cnn_f1,  'preds': cnn_pred,  'y': yi_test},
]

names = [r['name'] for r in results]
accs  = [r['accuracy'] for r in results]
f1s   = [r['f1'] for r in results]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - 0.2, accs, 0.35, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + 0.2, f1s,  0.35, label='F1 (macro)', color='coral')

for b, v in zip(list(bars1) + list(bars2), accs + f1s):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — ASL Letter Classification')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "model_comparison.png"), dpi=150)
plt.show()

best = max(results, key=lambda r: r['f1'])
print(f"\nBest model: {best['name']}  (Acc={best['accuracy']:.4f}  F1={best['f1']:.4f})")

## 7. Confusion Matrix for Best Model

In [ ]:
best = max(results, key=lambda r: r['f1'])
cm = confusion_matrix(best['y'], best['preds'])

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names,
            linewidths=0.3, ax=ax)
ax.set_title(f"{best['name']} — Confusion Matrix (Test set)\nAccuracy: {best['accuracy']:.4f}", fontsize=14)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, f"confusion_{best['name'].replace(' ', '_').lower()}.png"),
    dpi=150,
)
plt.show()

# Show most confused pairs
np.fill_diagonal(cm, 0)
top_errors = np.dstack(np.unravel_index(np.argsort(-cm.ravel()), cm.shape))[0][:10]
print('\nTop confused letter pairs (true → predicted):')
for true_idx, pred_idx in top_errors:
    if cm[true_idx, pred_idx] > 0:
        print(f'  {label_names[true_idx]} → {label_names[pred_idx]}: {cm[true_idx, pred_idx]} errors')